# CP2/CP3: ETH/USD next-minute close prediction

Ноутбук запускает тот же воспроизводимый pipeline, что и `scripts/train.py`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))
PROJECT_ROOT

## 1. Загрузка и очистка данных

In [ ]:
from eth_price.data import load_raw_data, clean_raw_data, describe_dataframe

raw = load_raw_data(PROJECT_ROOT / "data/sample/ethusd_1m_sample.csv", max_rows=10_000)
clean, cleaning_report = clean_raw_data(raw)
print(describe_dataframe(clean))
cleaning_report

## 2. Feature engineering и хронологический split

In [ ]:
from eth_price.features import make_features, feature_columns
from eth_price.data import train_val_test_split_by_time, assert_no_temporal_overlap

features = make_features(clean)
cols = feature_columns(features)
train, val, test = train_val_test_split_by_time(features)
assert_no_temporal_overlap([train, val, test])
print(features.shape, len(cols))

## 3. Полное обучение и сохранение артефактов

In [ ]:
from eth_price.train import run_training

summary = run_training(
    data_path=PROJECT_ROOT / "data/sample/ethusd_1m_sample.csv",
    max_rows=10_000,
    model_path=PROJECT_ROOT / "models/eth_next_close_model.joblib",
    metrics_path=PROJECT_ROOT / "report/experiment_results.csv",
    report_dir=PROJECT_ROOT / "report",
    figures_dir=PROJECT_ROOT / "report/figures",
)
summary

## 4. Таблица экспериментов

In [ ]:
import pandas as pd

pd.read_csv(PROJECT_ROOT / "report/experiment_results.csv")